[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/047.%20The%20Demand%20Forecasting%20-%20Moving%20Average%20%26%20Weighted%20MA%20Problem/P47-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 47. The Demand Forecasting Problem
## Tier 1: The Pen-and-Paper Method (Mathematical Formulation)

### Goal and workflow

This notebook develops and evaluates two moving-average methods for forecasting product demand. The calculation sequence is: compute a Simple Moving Average (SMA), compute a Weighted Moving Average (WMA), evaluate historical forecasts with error metrics, compare the methods, and examine how the window size affects backtest performance.

The method is intentionally transparent: each forecast is a direct calculation from a specified number of recent observations. The code implements these calculations so that each result can be checked against the formulas and, where useful, calculated by hand.

### Inputs and outputs

- **Input:** weekly historical demand for wireless headphones, measured in units.
- **Forecast outputs:** SMA and WMA estimates for a specified future week.
- **Evaluation outputs:** forecast errors and summary metrics (MAD, MSE, RMSE, and MAPE) over historical periods for which actual demand is available.
- **Comparison outputs:** tables and plots for the two methods and for the tested window sizes and weight sequences.

### Model assumptions and scope

The methods use only the most recent observations in a fixed-length window. The SMA assigns equal influence to each observation; the WMA assigns nonnegative weights that sum to one. In the WMA calculations, the first weight belongs to the most recent observation. Historical errors are evaluated by making each forecast using only demand from earlier periods.

These methods do not explicitly model trend, seasonality, promotions, stockouts, or other causes of demand changes. The WMA weights and window sizes in this notebook are specified for illustration and sensitivity analysis; the notebook does not optimize them.

### Example data

Wireless headphone demand over 12 weeks is:

```text
Weeks 1-12: [450, 380, 520, 610, 485, 390, 475, 550, 425, 480, 515, 460] units
```

The worked forecast uses a three-period window to estimate demand for week 13. Historical backtesting then compares forecasts with observed demand over periods for which the required earlier observations are available.

### What to examine

- How the choice of window size affects forecast responsiveness and smoothing.
- How the WMA's recency weights affect its estimates relative to the SMA.
- Whether a method has lower error on this particular historical backtest, and how the answer depends on the chosen metric.
- How the error plots reveal overforecasting and underforecasting in individual periods.

A lower historical error does not guarantee better future forecasts. Backtest results depend on the observed demand sequence and the evaluation periods, and the sensitivity analysis is not an independent test set.

## Mathematical Formulation

### Moving-average forecasts

Let $D_t$ denote the observed demand in period $t$. To forecast period $t+1$, each method uses the $n$ observations from periods $t, t-1, \ldots, t-n+1$. The window length $n$ determines how many past observations contribute to the forecast.

The Simple Moving Average assigns each of those observations the same influence:

$$
F_{t+1} = \frac{1}{n} \sum_{i=0}^{n-1} D_{t-i}
       = \frac{D_t + D_{t-1} + \cdots + D_{t-n+1}}{n}.
$$

Here, $F_{t+1}$ is the forecast for period $t+1$, $D_{t-i}$ is demand from one of the $n$ most recent observed periods, and $n$ is the window size. In the code, these calculations are performed by `simple_moving_average`.

The Weighted Moving Average allows the observations to have different influences:

$$
F_{t+1} = \sum_{i=0}^{n-1} w_i D_{t-i},
\qquad
\sum_{i=0}^{n-1} w_i = 1,
\qquad
w_i \geq 0 \quad (i=0,\ldots,n-1).
$$

The index convention matters: $w_0$ multiplies the most recent demand $D_t$, $w_1$ multiplies $D_{t-1}$, and so on. The weights sum to one so that the weighted average remains on the scale of demand. Nonnegative weights make the forecast a weighted average rather than an extrapolation outside the range of the selected observations. The code implements this calculation in `weighted_moving_average`.

### Forecast errors and accuracy metrics

For a historical period $t$ with an available forecast $F_t$, define the signed error as actual demand minus forecast demand:

$$
E_t = D_t - F_t.
$$

Thus, a positive error indicates underforecasting, and a negative error indicates overforecasting. For $m$ evaluated periods, the Mean Absolute Deviation (MAD) and Mean Squared Error (MSE) are:

$$
MAD = \frac{1}{m} \sum_{t=1}^{m} |E_t|,
\qquad
MSE = \frac{1}{m} \sum_{t=1}^{m} E_t^2.
$$

MAD is expressed in demand units and gives the average absolute error. MSE is expressed in squared demand units and gives relatively more influence to larger errors. The Root Mean Squared Error is $RMSE = \sqrt{MSE}$, which returns the measure to demand units.

When actual demand is nonzero, the Mean Absolute Percentage Error is:

$$
MAPE = \frac{100\%}{m} \sum_{t=1}^{m} \left|\frac{E_t}{D_t}\right|.
$$

MAPE expresses absolute error relative to actual demand. It is undefined for an individual period with zero actual demand; the implementation excludes such periods from the MAPE average. MAD, MSE, and RMSE use all evaluated periods. The function `calculate_forecast_errors` computes these metrics and returns the signed errors as well.

### Historical backtesting and computational work

A forecast for period $t$ must use observations available before period $t$. For an $n$-period moving average, the first evaluable period is $n+1$. The backtest therefore applies the same forecasting rule repeatedly, advancing one period at a time and comparing each forecast with that period's actual demand. With $m$ forecasts and a window of length $n$, direct calculation requires $O(mn)$ arithmetic operations for each method; this implementation favors the explicit calculation over a more compact update scheme.

In [ ]:
# NumPy supports the numerical calculations; Matplotlib and Seaborn format the plots.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported for moving-average forecasting analysis.")

In [ ]:
# The example demand observations are the input series for every calculation below.
demand_data = [450, 380, 520, 610, 485, 390, 475, 550, 425, 480, 515, 460]
weeks = list(range(1, 13))

print("=== Demand Data Summary ===")
print(f"Weekly demand data (units): {demand_data}")
print(f"Number of periods: {len(demand_data)}")
print(f"Mean demand: {np.mean(demand_data):.1f} units")
print(f"Population standard deviation: {np.std(demand_data):.1f} units")
print(f"Coefficient of variation: {np.std(demand_data) / np.mean(demand_data):.3f}")

In [ ]:
class MovingAverageForecaster:
    """Calculate simple and weighted moving-average forecasts and errors."""

    def __init__(self, demand_data: List[float]):
        """Store the historical demand series used by the forecasting methods."""
        if not demand_data:
            raise ValueError("Demand data must contain at least one observation.")

        self.demand_data = list(demand_data)
        self.periods = len(self.demand_data)

    def simple_moving_average(self, window_size: int, forecast_period: int) -> float:
        """Forecast a period using the arithmetic mean of the latest observations.

        Period numbers are one-based. A forecast for period ``p`` uses the
        observations from periods ``p - window_size`` through ``p - 1``.
        """
        if not isinstance(window_size, int) or window_size < 1:
            raise ValueError("Window size must be a positive integer.")
        if not isinstance(forecast_period, int):
            raise ValueError("Forecast period must be an integer.")
        if not window_size < forecast_period <= self.periods + 1:
            raise ValueError(
                "The forecast period must have enough preceding observations "
                "and cannot be more than one period beyond the data."
            )

        # Convert the one-based period number to the corresponding Python slice.
        start_idx = forecast_period - window_size - 1
        end_idx = forecast_period - 1
        recent_data = self.demand_data[start_idx:end_idx]
        return sum(recent_data) / window_size

    def weighted_moving_average(
        self,
        window_size: int,
        weights: List[float],
        forecast_period: int,
    ) -> float:
        """Forecast using weights ordered from most recent to oldest."""
        if not isinstance(window_size, int) or window_size < 1:
            raise ValueError("Window size must be a positive integer.")
        if len(weights) != window_size:
            raise ValueError(
                f"Number of weights ({len(weights)}) must equal "
                f"window size ({window_size})."
            )
        if any(weight < 0 for weight in weights):
            raise ValueError("Weights must be nonnegative.")
        if not np.isclose(sum(weights), 1.0, rtol=0.0, atol=1e-6):
            raise ValueError(f"Weights must sum to 1.0; current sum: {sum(weights)}.")
        if not isinstance(forecast_period, int):
            raise ValueError("Forecast period must be an integer.")
        if not window_size < forecast_period <= self.periods + 1:
            raise ValueError(
                "The forecast period must have enough preceding observations "
                "and cannot be more than one period beyond the data."
            )

        start_idx = forecast_period - window_size - 1
        end_idx = forecast_period - 1
        # The formula assigns the first weight to the most recent observation.
        recent_data = self.demand_data[start_idx:end_idx][::-1]
        return sum(weight * demand for weight, demand in zip(weights, recent_data))

    def calculate_forecast_errors(
        self,
        forecasts: List[float],
        start_period: int,
    ) -> Dict[str, float]:
        """Calculate error metrics for consecutive, one-based periods."""
        if not forecasts:
            raise ValueError("At least one forecast is required to calculate errors.")
        if not isinstance(start_period, int) or start_period < 1:
            raise ValueError("Start period must be a positive integer.")
        end_period = start_period + len(forecasts) - 1
        if end_period > self.periods:
            raise ValueError("Every forecast must have a corresponding actual value.")

        actual_values = self.demand_data[start_period - 1:end_period]
        if len(actual_values) != len(forecasts):
            raise ValueError("Forecasts and actual values must have equal lengths.")

        errors = [actual - forecast for actual, forecast in zip(actual_values, forecasts)]
        mad = float(np.mean(np.abs(errors)))
        mse = float(np.mean(np.square(errors)))
        rmse = float(np.sqrt(mse))

        # A percentage error is undefined when actual demand is zero.
        mape_values = [
            abs(error / actual) * 100
            for actual, error in zip(actual_values, errors)
            if actual != 0
        ]
        mape = float(np.mean(mape_values)) if mape_values else float("nan")

        return {
            "MAD": mad,
            "MSE": mse,
            "RMSE": rmse,
            "MAPE": mape,
            "errors": errors,
        }

print("MovingAverageForecaster class is ready.")

## A Three-Period Forecast for Week 13

The forecast uses the three most recent observations: weeks 10, 11, and 12. For the SMA, all three receive equal weight. For the WMA, the stated weights are ordered from most recent to oldest, so the first weight is applied to week 12. The calculations below show the arithmetic explicitly before calling the forecasting methods.

In [ ]:
forecaster = MovingAverageForecaster(demand_data)

window_size = 3
forecast_period = 13
wma_weights = [0.5, 0.3, 0.2]  # Weeks 12, 11, and 10, respectively.

print("=== Worked Forecast Example ===")
print(f"\n{window_size}-period Simple Moving Average for week {forecast_period}:")
print(
    f"F_{forecast_period} = "
    f"(D_{forecast_period - 1} + D_{forecast_period - 2} + "
    f"D_{forecast_period - 3}) / {window_size}"
)
sma_forecast = forecaster.simple_moving_average(window_size, forecast_period)
print(
    f"F_{forecast_period} = ({demand_data[11]} + {demand_data[10]} + "
    f"{demand_data[9]}) / {window_size} = {sma_forecast:.1f} units"
)

print(f"\n{window_size}-period Weighted Moving Average for week {forecast_period}:")
print(f"Weights (most recent to oldest): {wma_weights}")
wma_forecast = forecaster.weighted_moving_average(
    window_size, wma_weights, forecast_period
)
print(
    f"F_{forecast_period} = 0.5 * {demand_data[11]} + "
    f"0.3 * {demand_data[10]} + 0.2 * {demand_data[9]} "
    f"= {wma_forecast:.1f} units"
)
print(f"\nSMA minus WMA forecast: {sma_forecast - wma_forecast:.1f} units")

## Historical Backtest

A backtest applies the forecasting rule to earlier periods for which actual demand is known. For a three-period window, the first forecast is for period 4: it uses periods 1–3, then compares the forecast with period 4's actual demand. The same procedure advances through period 12. This chronological setup prevents a forecast from using the observation it is intended to predict.

The signed error is actual demand minus forecast demand. Positive errors therefore mean underforecasting, while negative errors mean overforecasting. Summary metrics condense these period-by-period errors, but each metric emphasizes a different aspect of accuracy.

In [ ]:
print("=== Historical Backtest: Periods 4-12 ===")

backtest_periods = list(range(4, 13))
sma_forecasts = [
    forecaster.simple_moving_average(3, period)
    for period in backtest_periods
]
wma_forecasts = [
    forecaster.weighted_moving_average(3, wma_weights, period)
    for period in backtest_periods
]

sma_errors = forecaster.calculate_forecast_errors(sma_forecasts, 4)
wma_errors = forecaster.calculate_forecast_errors(wma_forecasts, 4)

print("\nPeriod | Actual | SMA forecast | WMA forecast | SMA error | WMA error")
print("-" * 72)
for index, period in enumerate(backtest_periods):
    actual = demand_data[period - 1]
    sma_forecast_value = sma_forecasts[index]
    wma_forecast_value = wma_forecasts[index]
    sma_error = sma_errors["errors"][index]
    wma_error = wma_errors["errors"][index]
    print(
        f"{period:6d} | {actual:6.0f} | {sma_forecast_value:12.1f} | "
        f"{wma_forecast_value:12.1f} | {sma_error:9.1f} | {wma_error:9.1f}"
    )

print("\nBacktest error metrics:")
print(
    f"SMA - MAD: {sma_errors['MAD']:.2f} units, "
    f"MSE: {sma_errors['MSE']:.2f} units squared, "
    f"RMSE: {sma_errors['RMSE']:.2f} units, "
    f"MAPE: {sma_errors['MAPE']:.2f}%"
)
print(
    f"WMA - MAD: {wma_errors['MAD']:.2f} units, "
    f"MSE: {wma_errors['MSE']:.2f} units squared, "
    f"RMSE: {wma_errors['RMSE']:.2f} units, "
    f"MAPE: {wma_errors['MAPE']:.2f}%"
)

print("\nRelative change in error (positive means the WMA error is lower):")
mad_improvement = (
    (sma_errors["MAD"] - wma_errors["MAD"]) / sma_errors["MAD"] * 100
)
rmse_improvement = (
    (sma_errors["RMSE"] - wma_errors["RMSE"]) / sma_errors["RMSE"] * 100
)
mape_improvement = (
    (sma_errors["MAPE"] - wma_errors["MAPE"]) / sma_errors["MAPE"] * 100
)
print(f"MAD relative change: {mad_improvement:.1f}%")
print(f"RMSE relative change: {rmse_improvement:.1f}%")
print(f"MAPE relative change: {mape_improvement:.1f}%")

## Sensitivity Analysis

The window size controls how many observations enter each forecast. A shorter window responds more directly to recent demand; a longer window averages over more observations and can smooth short-term fluctuations. Neither effect guarantees lower forecast error.

This analysis evaluates window sizes 2, 3, 4, and 5 over the periods available for each window. For each WMA window, the weights decrease exponentially with age and are normalized to sum to one. Because different window sizes are evaluated over different starting periods, their error summaries should be interpreted with that difference in evaluation periods in mind. The lowest MAD identifies the best configuration among those tested on these historical data; it is not a guarantee of future performance.

In [ ]:
print("=== Sensitivity Analysis ===")

window_sizes = [2, 3, 4, 5]
results = []

for window in window_sizes:
    evaluation_periods = range(window + 1, len(demand_data) + 1)

    sma_forecasts_window = [
        forecaster.simple_moving_average(window, period)
        for period in evaluation_periods
    ]
    sma_window_errors = forecaster.calculate_forecast_errors(
        sma_forecasts_window, window + 1
    )

    # Exponential decay gives more recent observations greater influence.
    weights = [2 ** (-(window - 1 - index)) for index in range(window)]
    total_weight = sum(weights)
    weights = [weight / total_weight for weight in weights]

    wma_forecasts_window = [
        forecaster.weighted_moving_average(window, weights, period)
        for period in evaluation_periods
    ]
    wma_window_errors = forecaster.calculate_forecast_errors(
        wma_forecasts_window, window + 1
    )

    results.append(
        {
            "Window_Size": window,
            "SMA_MAD": sma_window_errors["MAD"],
            "SMA_MAPE": sma_window_errors["MAPE"],
            "WMA_MAD": wma_window_errors["MAD"],
            "WMA_MAPE": wma_window_errors["MAPE"],
            "Weights": weights,
        }
    )

print("\nSensitivity results:")
print("Window | SMA MAD | SMA MAPE | WMA MAD | WMA MAPE | Weights (most recent to oldest)")
print("-" * 88)
for result in results:
    weights_text = ", ".join(f"{weight:.3f}" for weight in result["Weights"])
    print(
        f"{result['Window_Size']:6d} | {result['SMA_MAD']:7.2f} | "
        f"{result['SMA_MAPE']:8.2f}% | {result['WMA_MAD']:7.2f} | "
        f"{result['WMA_MAPE']:8.2f}% | [{weights_text}]"
    )

lowest_wma_mad = min(result["WMA_MAD"] for result in results)
best_wma_configurations = [
    result
    for result in results
    if np.isclose(result["WMA_MAD"], lowest_wma_mad, rtol=1e-9, atol=1e-12)
]
print("\nWMA configuration(s) with the lowest MAD among those tested:")
for result in best_wma_configurations:
    print(
        f"Window size {result['Window_Size']}: "
        f"MAD = {result['WMA_MAD']:.2f} units"
    )

## Visualizing the Forecasts and Errors

The first panel places historical demand, the period-by-period backtest forecasts, and the week 13 forecasts on the same time axis. The error panel preserves the sign of each error, making underforecasting and overforecasting distinguishable. The metric panel compares summary measures, whose units differ: MAD and RMSE are in demand units, MAPE is a percentage, and the underlying MSE is not plotted. The final panel shows how backtest MAD varies with the window size.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Demand Forecasting Analysis: Moving-Average Methods", fontsize=16)

# Historical forecasts and the week 13 estimates share the demand time axis.
ax1 = axes[0, 0]
ax1.plot(weeks, demand_data, "o-", linewidth=2, markersize=7, label="Actual demand")
ax1.plot(
    backtest_periods,
    sma_forecasts,
    "s--",
    linewidth=1.8,
    markersize=5,
    label="3-period SMA forecast",
)
ax1.plot(
    backtest_periods,
    wma_forecasts,
    "^--",
    linewidth=1.8,
    markersize=5,
    label="3-period WMA forecast",
)
ax1.plot([13], [sma_forecast], "s", markersize=9, label="SMA forecast, week 13")
ax1.plot([13], [wma_forecast], "^", markersize=9, label="WMA forecast, week 13")
ax1.set_xlabel("Week")
ax1.set_ylabel("Demand (units)")
ax1.set_title("Actual Demand and Forecasts")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Signed errors make the direction of each forecast miss visible.
ax2 = axes[0, 1]
x_positions = np.arange(len(backtest_periods))
bar_width = 0.35
ax2.bar(
    x_positions - bar_width / 2,
    sma_errors["errors"],
    bar_width,
    label="SMA error",
)
ax2.bar(
    x_positions + bar_width / 2,
    wma_errors["errors"],
    bar_width,
    label="WMA error",
)
ax2.set_xlabel("Period")
ax2.set_ylabel("Actual minus forecast (units)")
ax2.set_title("Signed Backtest Errors")
ax2.set_xticks(x_positions)
ax2.set_xticklabels(backtest_periods)
ax2.axhline(y=0, color="black", linewidth=1, alpha=0.5)
ax2.legend()
ax2.grid(True, axis="y", alpha=0.3)

# MAD and RMSE are in units; MAPE is a percentage, so bar heights have different scales.
ax3 = axes[1, 0]
metrics = ["MAD (units)", "RMSE (units)", "MAPE (%)"]
sma_metric_values = [
    sma_errors["MAD"],
    sma_errors["RMSE"],
    sma_errors["MAPE"],
]
wma_metric_values = [
    wma_errors["MAD"],
    wma_errors["RMSE"],
    wma_errors["MAPE"],
]
metric_positions = np.arange(len(metrics))
ax3.bar(
    metric_positions - bar_width / 2,
    sma_metric_values,
    bar_width,
    label="SMA",
)
ax3.bar(
    metric_positions + bar_width / 2,
    wma_metric_values,
    bar_width,
    label="WMA",
)
ax3.set_xlabel("Error metric")
ax3.set_ylabel("Metric value (scales differ by metric)")
ax3.set_title("Backtest Error Metrics")
ax3.set_xticks(metric_positions)
ax3.set_xticklabels(metrics)
ax3.legend()
ax3.grid(True, axis="y", alpha=0.3)

# Sensitivity results show the observed MAD for each tested window size.
ax4 = axes[1, 1]
sensitivity_window_sizes = [result["Window_Size"] for result in results]
sma_mads = [result["SMA_MAD"] for result in results]
wma_mads = [result["WMA_MAD"] for result in results]
ax4.plot(
    sensitivity_window_sizes,
    sma_mads,
    "o-",
    linewidth=2,
    markersize=7,
    label="SMA MAD",
)
ax4.plot(
    sensitivity_window_sizes,
    wma_mads,
    "s-",
    linewidth=2,
    markersize=7,
    label="WMA MAD",
)
ax4.set_xlabel("Window size (periods)")
ax4.set_ylabel("Mean Absolute Deviation (units)")
ax4.set_title("Window-Size Sensitivity")
ax4.set_xticks(sensitivity_window_sizes)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Forecast and backtest plots are ready.")

## Results Analysis and Interpretation

### Reading the outputs

The worked example reports the week 13 SMA and WMA forecasts using the last three observations. The backtest table then shows each actual value, forecast, and signed error for the evaluated historical periods. Its summary metrics describe different error properties: MAD is the average absolute miss in units, RMSE gives greater influence to large misses, and MAPE expresses absolute errors relative to actual demand.

The printed relative change compares WMA error with SMA error for the same backtest periods and metric. A positive value means the WMA metric is lower on this sample; a negative value means it is higher. Such a comparison is descriptive of this demand history, not proof that one method will perform better in future periods.

In the sensitivity analysis, each window size is evaluated over all periods for which that window has enough preceding observations. The resulting MAD values therefore summarize different evaluation ranges as the window changes. The reported lowest-MAD configuration is only the best among the tested WMA configurations under this evaluation; ties are reported using a numerical tolerance.

### Modeling insights

The SMA and WMA are both averages of past demand, but they express recency differently. Equal SMA weights treat all observations in the window alike. The WMA's specified weights can emphasize recent observations, but this notebook does not choose weights by an optimization procedure. The requirement that weights sum to one preserves the scale of the averaged demand; nonnegative weights make the forecast a convex combination of the selected observations.

The error plots complement the summary metrics. A positive signed error is an underforecast, and a negative signed error is an overforecast. MAD and RMSE ignore this direction by using absolute or squared errors, so the signed-error chart is needed to inspect whether forecast misses tend to occur in a particular direction.

### Scope and limitations

Moving averages provide a transparent baseline for demand forecasting and are useful when a simple, reproducible calculation is appropriate. Their fixed windows and weights do not explicitly represent trend, seasonality, changing demand conditions, or operational causes such as promotions and stockouts. Performance also depends on the selected window, weights, evaluation period, and error metric. These limitations define the scope of the method examined here.